# Phase 8 — Planning & Reasoning

> **Gemini-only project.** Every cell here uses Google Gemini (chat + embeddings) via `langchain-google-genai`. The only key the core needs is `GOOGLE_API_KEY`. This notebook is a **scaffold** — run it top-to-bottom *after* Claude Code finishes this phase. If a cell references a module that doesn't exist yet, that phase hasn't been built.

**Goal:** a complex query is decomposed into a plan BEFORE agents run; synthesis handles conflicts.


In [ ]:
# --- Setup: load .env and make the project importable ---
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # repo root
from dotenv import load_dotenv
load_dotenv('../.env', override=True)  # override=True: beat VS Code's stale cached env vars

# This project is Gemini-only. Confirm the one required key is present.
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env (see 00_prerequisites_and_setup.md)'
print('GOOGLE_API_KEY loaded:', bool(os.getenv('GOOGLE_API_KEY')))
print('SEC_USER_AGENT   :', os.getenv('SEC_USER_AGENT', '(set before Phase 5)'))

## 1. Planner decomposes a compound query

In [ ]:
from app.graph.planner import PlannerNode
state = {'messages': [('user', 'Should I rebalance given the recent Fed announcement and my risk tolerance?')],
         'client_id': 'CLT-005', 'session_id': 'nb-8'}
planned = PlannerNode().run(state)
print('PLAN (printed before any agent runs):')
for i, step in enumerate(planned['plan'], 1):
    print(f'  {i}. {step}')

## 2. Full run through the graph (plan -> agents -> synthesizer)

In [ ]:
from app.graph.builder import GraphBuilder
graph = GraphBuilder().with_all().build()
out = graph.invoke(state, {'configurable': {'thread_id': 'CLT-005-nb8'}})
print(out['messages'][-1].content)

## 3. Conflicting information is surfaced, not hidden

In [ ]:
conflict_q = {'messages': [('user', 'News is bullish on NVDA but momentum looks weak - what do I do?')],
              'client_id': 'CLT-005', 'session_id': 'nb-8b'}
out = graph.invoke(conflict_q, {'configurable': {'thread_id': 'CLT-005-nb8b'}})
print(out['messages'][-1].content)  # should acknowledge the conflict explicitly

## ✅ Acceptance check

In [ ]:
assert len(planned['plan']) >= 2, 'plan should have multiple steps'
print('Phase 8 acceptance: PASS (plan printed before execution)')